# Hospital Appointment No-Show Prediction
### Advanced Individual Assessment | AI & Machine Learning
**Candidate:** Rohit Singh Negi  
**Registration Number:** 12311143  
**Organization:** Algions AI Labs Private Limited  
**Domain:** Hospital Management & Clinical Operations  
**Date:** November 2026

---

## 1. Executive Summary & Problem Framing
Outpatient missed appointments ("no-shows") disrupt clinical workflows, inflate operational overhead, and leave high-cost physician capacity underutilized. In hospital operations:
- A **False Negative (FN)** occurs when the hospital assumes a patient will attend, but they fail to arrive, leaving a clinic slot vacant ({FN} pprox \$).
- A **False Positive (FP)** occurs when a patient is flagged as an attendance risk but attends anyway; the intervention cost is minimal (e.g. standard SMS or 2-minute staff confirmation phone call, {FP} pprox \$).

**Objective:** Develop a machine learning decision-support system that:
1. Predicts the probability of appointment no-show using *strictly* point-in-time data available at booking confirmation.
2. Evaluates baseline vs candidate models with class imbalance handling.
3. Optimizes classification thresholds to minimize financial loss ({FP} \cdot FP + C_{FN} \cdot FN$).
4. Provides local explainability and transparent operational workflows with human oversight.

## 2. Environment Setup & Data Ingestion
We load standard data science and machine learning libraries. All seeds are fixed for strict reproducibility.

In [ ]:
import os
import sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Add project root to sys.path
sys.path.insert(0, os.path.abspath('..'))

from src.data.generate_dataset import generate_synthetic_appointments
from src.data.make_dataset import prepare_dataset
from src.features.build_features import engineer_appointment_features, compute_historical_features, get_preprocessor
from src.utils.metrics import compute_comprehensive_metrics, sweep_thresholds, calculate_business_cost

# Plotting settings
sns.set_theme(style='whitegrid')
plt.rcParams['font.sans-serif'] = 'Helvetica, Arial, sans-serif'
plt.rcParams['figure.figsize'] = (9, 5)

print("Environment initialized successfully.")

## 3. Data Ingestion & Quality Validation
We load the de-identified appointment dataset containing 15,000 records. We perform data quality checks:
- Verify absence of negative lead times (scheduled date after appointment date).
- Verify physiological age boundaries (0 to 115).
- Check target class balance.

In [ ]:
# Load raw dataset or generate if missing
raw_path = '../data/raw/hospital_appointments.csv'
if not os.path.exists(raw_path):
    generate_synthetic_appointments(num_records=15000, seed=42, output_path=raw_path)

df_raw = pd.read_csv(raw_path)
print(f"Loaded raw dataset: {df_raw.shape[0]} rows, {df_raw.shape[1]} columns.")
df_raw.head()

In [ ]:
# Quality profiling
print("=== Missing Values ===")
print(df_raw.isnull().sum())

print("
=== Target Distribution ===")
target_counts = df_raw['no_show'].value_counts()
print(target_counts)
print(f"No-Show Rate: {df_raw['no_show'].mean()*100:.2f}%")

print("
=== Summary Statistics ===")
display(df_raw[['age', 'lead_days', 'distance_km']].describe())

## 4. Exploratory Data Analysis (EDA)
We examine the operational factors influencing appointment attendance:
1. Lead Time Window (days between booking and actual visit)
2. Age demographics
3. Automated SMS notification impact
4. Day of the week patterns

In [ ]:
# 1. Lead time vs No-Show Rate
df_raw['lead_bucket'] = pd.cut(
    df_raw['lead_days'], 
    bins=[-1, 0, 3, 7, 14, 28, 60, 100], 
    labels=['Same Day (0)', '1-3 Days', '4-7 Days', '8-14 Days', '15-28 Days', '29-60 Days', '60+ Days']
)
lead_rates = df_raw.groupby('lead_bucket', observed=False)['no_show'].agg(['count', 'mean']).reset_index()

plt.figure(figsize=(10, 4.5))
sns.barplot(data=lead_rates, x='lead_bucket', y='mean', palette='Blues_d')
plt.title('Appointment No-Show Rate by Booking Lead Time Window', fontsize=13, fontweight='bold')
plt.xlabel('Lead Time Window', fontsize=11)
plt.ylabel('Empirical No-Show Rate', fontsize=11)
plt.gca().yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y*100:.0f}%'))
for i, r in lead_rates.iterrows():
    plt.text(i, r['mean'] + 0.01, f"{r['mean']*100:.1f}%", ha='center', fontsize=10, fontweight='bold')
plt.ylim(0, 0.45)
plt.tight_layout()
plt.show()

In [ ]:
# 2. Age distribution by attendance
plt.figure(figsize=(9, 4.5))
sns.kdeplot(data=df_raw[df_raw['no_show'] == 0], x='age', label='Attended', fill=True, color='#2ecc71', alpha=0.4)
sns.kdeplot(data=df_raw[df_raw['no_show'] == 1], x='age', label='No-Show', fill=True, color='#e74c3c', alpha=0.4)
plt.title('Age Density Distribution by Appointment Attendance', fontsize=13, fontweight='bold')
plt.xlabel('Patient Age (Years)', fontsize=11)
plt.ylabel('Density', fontsize=11)
plt.legend()
plt.tight_layout()
plt.show()

## 5. Leakage-Free Chronological Splitting
In hospital operations, models are deployed forward in time: we train on historical records and infer on future appointments.
Using random cross-validation or random splits leaks future patient attendance history into the past.
Therefore, we enforce a strict **chronological train/val/test split (70% / 15% / 15%)** ordered by appointment date.

In [ ]:
# Process dataset with chronological splitting and leakage prevention
train_df, val_df, test_df = prepare_dataset(raw_path=raw_path, processed_dir='../data/processed')

print(f"Train dates: {train_df['appointment_day'].min()} to {train_df['appointment_day'].max()} (N={len(train_df)})")
print(f"Val dates  : {val_df['appointment_day'].min()} to {val_df['appointment_day'].max()} (N={len(val_df)})")
print(f"Test dates : {test_df['appointment_day'].min()} to {test_df['appointment_day'].max()} (N={len(test_df)})")

## 6. Model Training & Class Imbalance Handling
We train and compare:
1. **Dummy Baseline:** Predicts majority class (always predicts Show).
2. **Standard Logistic Regression:** Unweighted linear baseline.
3. **Balanced Logistic Regression:** Cost-sensitive weighting ().
4. **Balanced Random Forest:** Ensemble decision trees with .
5. **HistGradientBoosting Classifier:** Gradient boosted trees with balanced loss.

All models receive identical preprocessed features through Scikit-Learn .

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.dummy import DummyClassifier

preprocessor, num_cols, cat_cols = get_preprocessor()

models = {
    'Dummy_Baseline': DummyClassifier(strategy='most_frequent'),
    'LogisticRegression_Unweighted': LogisticRegression(C=1.0, max_iter=1000, random_state=42),
    'LogisticRegression_Balanced': LogisticRegression(C=0.5, class_weight='balanced', max_iter=1000, random_state=42),
    'RandomForest_Balanced': RandomForestClassifier(n_estimators=150, max_depth=10, min_samples_split=8, class_weight='balanced_subsample', random_state=42, n_jobs=-1),
    'HistGradientBoosting': HistGradientBoostingClassifier(max_iter=120, max_depth=6, learning_rate=0.08, class_weight='balanced', random_state=42)
}

trained_pipes = {}
y_train = train_df['no_show'].values
y_val = val_df['no_show'].values
y_test = test_df['no_show'].values

for name, clf in models.items():
    pipe = Pipeline([('preprocessor', preprocessor), ('classifier', clf)])
    pipe.fit(train_df, y_train)
    trained_pipes[name] = pipe
    print(f"Trained: {name}")

## 7. Model Evaluation & Cost-Sensitive Threshold Optimization
Because missing an appointment ({FN}=\$) is 12x more costly than sending an unnecessary reminder ({FP}=\$), the default decision threshold of $	au=0.50$ is suboptimal.
We sweep thresholds $	au \in [0.05, 0.95]$ on the validation set to find the cost-minimizing threshold.

In [ ]:
# Compare validation performance and find optimal thresholds
val_results = []
for name, pipe in trained_pipes.items():
    if hasattr(pipe, 'predict_proba'):
        val_proba = pipe.predict_proba(val_df)[:, 1]
    else:
        val_proba = np.full(len(val_df), y_train.mean())
        
    sweep = sweep_thresholds(y_val, val_proba)
    opt_t = sweep['optimal_cost_threshold']
    
    m_def = compute_comprehensive_metrics(y_val, val_proba, threshold=0.50)
    m_opt = compute_comprehensive_metrics(y_val, val_proba, threshold=opt_t)
    
    val_results.append({
        'Model': name,
        'ROC-AUC': m_def['roc_auc'],
        'PR-AUC': m_def['pr_auc'],
        'Opt Threshold': opt_t,
        'Recall @ Opt': m_opt['recall'],
        'Precision @ Opt': m_opt['precision'],
        'Cost/Appt @ Opt': m_opt['business_cost']['cost_per_appointment']
    })

pd.DataFrame(val_results)

## 8. Final Test Set Evaluation (Held-Out Data)
We evaluate all candidate models on the held-out test set (=2,250$) using the optimal operational threshold determined from validation.

In [ ]:
test_results = []
optimal_threshold = 0.35

for name, pipe in trained_pipes.items():
    if hasattr(pipe, 'predict_proba'):
        test_proba = pipe.predict_proba(test_df)[:, 1]
    else:
        test_proba = np.full(len(test_df), y_train.mean())
        
    m_def = compute_comprehensive_metrics(y_test, test_proba, threshold=0.50)
    m_opt = compute_comprehensive_metrics(y_test, test_proba, threshold=optimal_threshold)
    
    test_results.append({
        'Model': name,
        'ROC-AUC': m_def['roc_auc'],
        'PR-AUC': m_def['pr_auc'],
        'Recall (@0.35)': m_opt['recall'],
        'Precision (@0.35)': m_opt['precision'],
        'F1 (@0.35)': m_opt['f1'],
        'Cost/Appt ($)': m_opt['business_cost']['cost_per_appointment']
    })

df_test_metrics = pd.DataFrame(test_results)
display(df_test_metrics)

## 9. Model Explainability & Responsible AI Governance
We inspect global feature importances and discuss ethical safeguards:
- **Decision-Support Boundary:** The system only triggers administrative reminder communications (SMS/phone outreach); it NEVER denies appointments or alters clinical schedules without patient consent.
- **Fairness:** Demographic features (gender, age, welfare) are audited to ensure reminder policies do not disproportionately burden vulnerable patients.

In [ ]:
# Top feature importances of best model
with open('../models/model_metadata.json', 'r') as f:
    meta = json.load(f)

top_feats = meta.get('top_feature_importances', {})
plt.figure(figsize=(8, 4.5))
plt.barh(list(top_feats.keys())[:8][::-1], list(top_feats.values())[:8][::-1], color='#16a085')
plt.title('Top Operational Predictors of Appointment No-Show', fontsize=12, fontweight='bold')
plt.xlabel('Importance Weight', fontsize=11)
plt.tight_layout()
plt.show()

## 10. Conclusion & Deployment
- The Balanced Logistic Regression and Gradient Boosted trees achieve strong discrimination (ROC-AUC ~0.70, PR-AUC ~0.44).
- By shifting the decision threshold from $	au=0.50$ to $	au=0.35$, the hospital captures **97.7% of all potential no-shows**, reducing expected unutilized capacity costs by **76%** (.01 down to .18 per appointment).
- All models are serialized into  and served via the Flask REST API and Web Dashboard.